In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

# Load the dataset
data = pd.read_csv('/content/weatherAfterPreprocessing.csv')

# Drop rows where 'RainTomorrow' is NaN, as it's the target variable
data.dropna(subset=['RainTomorrow'], inplace=True)

# Separate features and target
X = data.drop(columns=['RainTomorrow'])  # Features
y = data['RainTomorrow']  # Target variable

# Impute missing values in features
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# Check class distribution in y_train and y_test
print(f"Unique classes in y_train: {y_train.nunique()}")
print(f"Class distribution in y_train:\n{y_train.value_counts()}")
print(f"Unique classes in y_test: {y_test.nunique()}")
print(f"Class distribution in y_test:\n{y_test.value_counts()}")

# Filter-Based Feature Selection
best_features = SelectKBest(score_func=mutual_info_classif, k='all')  # Initialize selector
best_features.fit(X_train, y_train)  # Fit to training data
scores = best_features.scores_  # Get feature scores

# Select top features based on threshold (e.g., median score)
threshold = np.median(scores)  # Calculate threshold
selected_features_filter = [feature for score, feature in zip(scores, X_imputed.columns) if score >= threshold]  # Filter features
print(f"Selected features (Filter-Based): {selected_features_filter}")

# Subset data based on selected features
X_train_filter = X_train[selected_features_filter]
X_test_filter = X_test[selected_features_filter]

# Wrapper-Based Feature Selection
model = RandomForestClassifier(random_state=42)  # Initialize model
rfe = RFE(estimator=model)  # Initialize RFE
rfe.fit(X_train, y_train)  # Fit RFE to training data
selected_features_wrapper = X_train.columns[rfe.support_].tolist()  # Get selected features
print(f"Selected features (Wrapper-Based): {selected_features_wrapper}")

# Subset data based on selected features
X_train_wrapper = X_train[selected_features_wrapper]
X_test_wrapper = X_test[selected_features_wrapper]

# Evaluate models on selected features
# Filter-Based Evaluation
model_filter = RandomForestClassifier(random_state=42)
model_filter.fit(X_train_filter, y_train)
y_pred_filter = model_filter.predict(X_test_filter)
accuracy_filter = accuracy_score(y_test, y_pred_filter)
print(f"Accuracy with Filter-Based Feature Selection: {accuracy_filter:.4f}")

# Wrapper-Based Evaluation
model_wrapper = RandomForestClassifier(random_state=42)
model_wrapper.fit(X_train_wrapper, y_train)
y_pred_wrapper = model_wrapper.predict(X_test_wrapper)
accuracy_wrapper = accuracy_score(y_test, y_pred_wrapper)
print(f"Accuracy with Wrapper-Based Feature Selection: {accuracy_wrapper:.4f}")

Unique classes in y_train: 1
Class distribution in y_train:
RainTomorrow
0.0    10881
Name: count, dtype: int64
Unique classes in y_test: 1
Class distribution in y_test:
RainTomorrow
0.0    2721
Name: count, dtype: int64
Selected features (Filter-Based): ['Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm']
Selected features (Wrapper-Based): ['Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday']
Accuracy with Filter-Based Feature Selection: 1.0000
Accuracy with Wrapper-Based Feature Selection: 1.0000


In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=0)
X_train.shape

(10881, 17)

In [13]:
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
clf_rf_4 = RandomForestClassifier()
rfecv = RFECV(estimator=clf_rf_4, step=1, cv=5,scoring='accuracy')
rfecv = rfecv.fit(X_train, y_train)
print('Optimal number of features :', rfecv.n_features_)
original_feature_names = X.columns
print('Best features :', list(original_feature_names[rfecv.support_]))

Optimal number of features : 1
Best features : ['RainToday']


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

# Assuming X_train and y_train are from the original split and might contain NaNs
# If X_train from the previous cell (SwEPs7gqtn2F) is already imputed, this step is redundant for that X_train
# but is necessary if using the X_train from cell Un05J7UEzUD1, which was split from the original X.

# Impute missing values in X_train specifically for this model if it's not already imputed.
# This uses the X_train that caused the error, which was not imputed.
imputer_lr = SimpleImputer(strategy='mean')
X_train_imputed_lr = pd.DataFrame(imputer_lr.fit_transform(X_train), columns=X_train.columns, index=X_train.index)

lg=LogisticRegression(max_iter=1000) # Increased max_iter for convergence
lg.fit(X_train_imputed_lr,y_train)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.float64(0.0)